In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE
from catboost import CatBoostClassifier, Pool
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

import optuna

pd.set_option("display.max_columns", None)

In [2]:
path_to_repo = Path('..').resolve()
path_to_data = path_to_repo / 'data'

In [3]:
df = pd.read_csv(path_to_data / 'preprocessed_data.csv')

In [4]:
y = df["bad"]
X = df.drop(columns=["bad", "ID"])

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# RANDOM FOREST CLASSIFIER

In [6]:
rf_base = RandomForestClassifier(random_state=42)
%time rf_base.fit(X_train, y_train)

CPU times: user 944 ms, sys: 9.37 ms, total: 953 ms
Wall time: 955 ms


RandomForestClassifier(random_state=42)

In [7]:
y_pred_default = rf_base.predict(X_test)

In [8]:
print(classification_report(y_test, y_pred_default))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99      7169
           1       0.33      0.14      0.19       123

    accuracy                           0.98      7292
   macro avg       0.66      0.57      0.59      7292
weighted avg       0.97      0.98      0.98      7292



In [9]:
rf_default_balanced = RandomForestClassifier(random_state=42,
                                    class_weight='balanced')
%time rf_default_balanced.fit(X_train, y_train)

CPU times: user 962 ms, sys: 8.67 ms, total: 971 ms
Wall time: 970 ms


RandomForestClassifier(class_weight='balanced', random_state=42)

In [10]:
y_pred_default_balanced = rf_default_balanced.predict(X_test)
print(classification_report(y_test, y_pred_default))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99      7169
           1       0.33      0.14      0.19       123

    accuracy                           0.98      7292
   macro avg       0.66      0.57      0.59      7292
weighted avg       0.97      0.98      0.98      7292



In [11]:
rf_default_balanced_subsample = RandomForestClassifier(random_state=42,
                                    class_weight='balanced_subsample')
%time rf_default_balanced_subsample.fit(X_train, y_train)

CPU times: user 1.15 s, sys: 11.1 ms, total: 1.16 s
Wall time: 1.16 s


RandomForestClassifier(class_weight='balanced_subsample', random_state=42)

In [12]:
y_pred_default_balanced_subsample = rf_default_balanced_subsample.predict(X_test)
print(classification_report(y_test, y_pred_default))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99      7169
           1       0.33      0.14      0.19       123

    accuracy                           0.98      7292
   macro avg       0.66      0.57      0.59      7292
weighted avg       0.97      0.98      0.98      7292



## RF + Hyperparameter Tuning

In [ ]:
def objective_rf(trial, X_train, y_train, X_val, y_val):
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 5, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 15)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    class_weight = trial.suggest_categorical("class_weight",["balanced", "balanced_subsample"])

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        class_weight=class_weight,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    f1 = f1_score(y_val, y_pred)
    return f1


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

In [15]:
study_rf = optuna.create_study(
    study_name="rf_opt",
    direction="maximize",
    storage="sqlite:///rf_opt.db",
    load_if_exists=True
)

study_rf.optimize(lambda trial: objective_rf(trial, X_train, y_train, X_val, y_val),n_trials=50, show_progress_bar=True)

[I 2025-12-06 18:15:56,935] Using an existing study with name 'rf_opt' instead of creating a new one.
Best trial: 11. Best value: 0.268293:   2%|▏         | 1/50 [00:00<00:29,  1.67it/s]

[I 2025-12-06 18:15:57,546] Trial 50 finished with value: 0.25806451612903225 and parameters: {'n_estimators': 219, 'max_depth': 30, 'min_samples_split': 10, 'min_samples_leaf': 4, 'class_weight': 'balanced'}. Best is trial 11 with value: 0.2682926829268293.


Best trial: 51. Best value: 0.270042:   4%|▍         | 2/50 [00:01<00:34,  1.40it/s]

[I 2025-12-06 18:15:58,341] Trial 51 finished with value: 0.270042194092827 and parameters: {'n_estimators': 250, 'max_depth': 16, 'min_samples_split': 14, 'min_samples_leaf': 2, 'class_weight': 'balanced_subsample'}. Best is trial 51 with value: 0.270042194092827.


Best trial: 51. Best value: 0.270042:   6%|▌         | 3/50 [00:02<00:32,  1.45it/s]

...

[I 2025-12-06 18:16:29,297] Trial 97 finished with value: 0.26556016597510373 and parameters: {'n_estimators': 226, 'max_depth': 16, 'min_samples_split': 15, 'min_samples_leaf': 2, 'class_weight': 'balanced_subsample'}. Best is trial 51 with value: 0.270042194092827.


Best trial: 51. Best value: 0.270042:  98%|█████████▊| 49/50 [00:33<00:00,  1.37it/s]

[I 2025-12-06 18:16:30,083] Trial 98 finished with value: 0.2540983606557377 and parameters: {'n_estimators': 245, 'max_depth': 22, 'min_samples_split': 14, 'min_samples_leaf': 3, 'class_weight': 'balanced_subsample'}. Best is trial 51 with value: 0.270042194092827.


Best trial: 51. Best value: 0.270042: 100%|██████████| 50/50 [00:33<00:00,  1.49it/s]

[I 2025-12-06 18:16:30,517] Trial 99 finished with value: 0.2549800796812749 and parameters: {'n_estimators': 212, 'max_depth': 17, 'min_samples_split': 13, 'min_samples_leaf': 7, 'class_weight': 'balanced'}. Best is trial 51 with value: 0.270042194092827.


In [16]:
print("Best F1:", study_rf.best_value)
print("Best params:", study_rf.best_params)

Best F1: 0.270042194092827
Best params: {'n_estimators': 250, 'max_depth': 16, 'min_samples_split': 14, 'min_samples_leaf': 2, 'class_weight': 'balanced_subsample'}


In [17]:
best_params = study_rf.best_params

best_rf = RandomForestClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X_train, y_train)

y_pred_best = best_rf.predict(X_test)

print(classification_report(y_test, y_pred_best))

              precision    recall  f1-score   support

           0       0.99      0.97      0.98      5376
           1       0.18      0.34      0.23        93

    accuracy                           0.96      5469
   macro avg       0.58      0.66      0.61      5469
weighted avg       0.97      0.96      0.97      5469



## RF + SMOTE

In [18]:
def objective_rf_smote(trial, X_train, y_train, X_val, y_val):
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 5, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    class_weight = trial.suggest_categorical("class_weight",
                                             ["balanced", "balanced_subsample"])

    smote = SMOTE(sampling_strategy=0.3, random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        class_weight=class_weight,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_res, y_train_res)

    y_pred = model.predict(X_val)

    f1 = f1_score(y_val, y_pred)

    return f1


In [19]:
study_rf_smote = optuna.create_study(
    study_name="rf_smote_opt",
    direction="maximize",
    storage="sqlite:///rf_smote.db",
    load_if_exists=True
)

[I 2025-12-06 18:16:31,408] Using an existing study with name 'rf_smote_opt' instead of creating a new one.


In [20]:
study_rf_smote.optimize(lambda trial: objective_rf_smote(trial, X_train, y_train, X_val, y_val), n_trials=50, show_progress_bar=True)

Best trial: 31. Best value: 0.25641:   2%|▏         | 1/50 [00:00<00:27,  1.79it/s]

[I 2025-12-06 18:16:31,982] Trial 50 finished with value: 0.2510822510822511 and parameters: {'n_estimators': 170, 'max_depth': 26, 'min_samples_split': 12, 'min_samples_leaf': 1, 'class_weight': 'balanced'}. Best is trial 31 with value: 0.2564102564102564.


Best trial: 51. Best value: 0.256637:   4%|▍         | 2/50 [00:01<00:26,  1.84it/s]

[I 2025-12-06 18:16:32,514] Trial 51 finished with value: 0.25663716814159293 and parameters: {'n_estimators': 184, 'max_depth': 30, 'min_samples_split': 12, 'min_samples_leaf': 1, 'class_weight': 'balanced'}. Best is trial 51 with value: 0.25663716814159293.


Best trial: 51. Best value: 0.256637:   6%|▌         | 3/50 [00:01<00:25,  1.84it/s]

...

[I 2025-12-06 18:16:59,112] Trial 97 finished with value: 0.242914979757085 and parameters: {'n_estimators': 198, 'max_depth': 27, 'min_samples_split': 12, 'min_samples_leaf': 2, 'class_weight': 'balanced'}. Best is trial 88 with value: 0.2636363636363636.


Best trial: 88. Best value: 0.263636:  98%|█████████▊| 49/50 [00:28<00:00,  1.62it/s]

[I 2025-12-06 18:16:59,782] Trial 98 finished with value: 0.25 and parameters: {'n_estimators': 237, 'max_depth': 30, 'min_samples_split': 13, 'min_samples_leaf': 1, 'class_weight': 'balanced'}. Best is trial 88 with value: 0.2636363636363636.


Best trial: 88. Best value: 0.263636: 100%|██████████| 50/50 [00:28<00:00,  1.73it/s]

[I 2025-12-06 18:17:00,287] Trial 99 finished with value: 0.23673469387755103 and parameters: {'n_estimators': 175, 'max_depth': 28, 'min_samples_split': 11, 'min_samples_leaf': 2, 'class_weight': 'balanced'}. Best is trial 88 with value: 0.2636363636363636.


In [21]:
best_params = study_rf_smote.best_params

smote = SMOTE(sampling_strategy=0.3, random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

final_rf_smote = RandomForestClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1
)

final_rf_smote.fit(X_train_res, y_train_res)

RandomForestClassifier(class_weight='balanced', max_depth=27,
                       min_samples_split=11, n_estimators=212, n_jobs=-1,
                       random_state=42)

In [22]:
y_pred_test = final_rf_smote.predict(X_test)
print(classification_report(y_test, y_pred_test))


              precision    recall  f1-score   support

           0       0.99      0.98      0.98      5376
           1       0.20      0.32      0.25        93

    accuracy                           0.97      5469
   macro avg       0.59      0.65      0.62      5469
weighted avg       0.97      0.97      0.97      5469



# XGBOOST CLASSIFIER

## XGBOOST + Hyperparameter Tuning

In [23]:
def objective_xgb(trial, X_train, y_train, X_val, y_val):

    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 30),        
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 50),
        "eval_metric": "logloss",
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)

    y_pred_val = model.predict(X_val)

    return f1_score(y_val, y_pred_val)

In [24]:
study_xgb = optuna.create_study(
    study_name="xgb_class_opt",
    direction="maximize",
    storage="sqlite:///xgb_class.db",
    load_if_exists=True
)

study_xgb.optimize(lambda trial: objective_xgb(trial, X_train, y_train, X_val, y_val), n_trials=50, show_progress_bar=True)

[I 2025-12-06 18:17:00,963] Using an existing study with name 'xgb_class_opt' instead of creating a new one.
Best trial: 42. Best value: 0.333333:   2%|▏         | 1/50 [00:01<01:01,  1.26s/it]

[I 2025-12-06 18:17:02,220] Trial 50 finished with value: 0.18181818181818182 and parameters: {'learning_rate': 0.12562413296885433, 'max_depth': 12, 'n_estimators': 718, 'subsample': 0.6739079447971166, 'colsample_bytree': 0.9730762034991742, 'gamma': 0.9809132797695113, 'min_child_weight': 22, 'scale_pos_weight': 2.4751486221453414}. Best is trial 42 with value: 0.3333333333333333.


Best trial: 42. Best value: 0.333333:   4%|▍         | 2/50 [00:02<01:00,  1.27s/it]

[I 2025-12-06 18:17:03,496] Trial 51 finished with value: 0.288135593220339 and parameters: {'learning_rate': 0.1029164612650194, 'max_depth': 12, 'n_estimators': 648, 'subsample': 0.7526684823725907, 'colsample_bytree': 0.9364456011115828, 'gamma': 1.634654620656382, 'min_child_weight': 20, 'scale_pos_weight': 11.087677922020028}. Best is trial 42 with value: 0.3333333333333333.


Best trial: 42. Best value: 0.333333:   6%|▌         | 3/50 [00:03<01:02,  1.33s/it]

...

[I 2025-12-06 18:17:56,295] Trial 97 finished with value: 0.2730627306273063 and parameters: {'learning_rate': 0.16328574586061534, 'max_depth': 12, 'n_estimators': 535, 'subsample': 0.6855553083069986, 'colsample_bytree': 0.7927472542652727, 'gamma': 1.8875373911004507, 'min_child_weight': 29, 'scale_pos_weight': 26.444682593876333}. Best is trial 88 with value: 0.34782608695652173.


Best trial: 88. Best value: 0.347826:  98%|█████████▊| 49/50 [00:56<00:00,  1.07it/s]

[I 2025-12-06 18:17:57,045] Trial 98 finished with value: 0.3 and parameters: {'learning_rate': 0.22426457336905006, 'max_depth': 12, 'n_estimators': 520, 'subsample': 0.6304868864554481, 'colsample_bytree': 0.7210805130119805, 'gamma': 3.355926968531908, 'min_child_weight': 27, 'scale_pos_weight': 8.364542758110433}. Best is trial 88 with value: 0.34782608695652173.


Best trial: 88. Best value: 0.347826: 100%|██████████| 50/50 [00:56<00:00,  1.14s/it]

[I 2025-12-06 18:17:57,751] Trial 99 finished with value: 0.24324324324324326 and parameters: {'learning_rate': 0.19313588770277468, 'max_depth': 12, 'n_estimators': 349, 'subsample': 0.6786572402108736, 'colsample_bytree': 0.7607853278014995, 'gamma': 1.0414763170317491, 'min_child_weight': 24, 'scale_pos_weight': 4.2964731896364885}. Best is trial 88 with value: 0.34782608695652173.


In [25]:
best_params = study_xgb.best_params

final_xgb = XGBClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss",
    tree_method="hist"
)

final_xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.783349756469322, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', feature_types=None, feature_weights=None,
              gamma=1.8419740427496967, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.2022240128652095,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=12, max_leaves=None,
              min_child_weight=23, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=478, n_jobs=-1,
              num_parallel_tree=None, ...)

In [26]:
y_pred_test = final_xgb.predict(X_test)
print(classification_report(y_test, y_pred_test))

              precision    recall  f1-score   support

           0       0.99      0.98      0.98      5376
           1       0.21      0.29      0.24        93

    accuracy                           0.97      5469
   macro avg       0.60      0.64      0.61      5469
weighted avg       0.97      0.97      0.97      5469



In [27]:
probs = final_xgb.predict_proba(X_test)[:, 1]

In [28]:
thresholds = np.linspace(0.01, 0.99, 200)

results = []

best_threshold = 0
best_f1 = 0

for t in thresholds:
    preds = (probs >= t).astype(int)
    f1 = f1_score(y_test, preds)
    precision = precision_score(y_test, preds, zero_division=0)
    recall = recall_score(y_test, preds)

    results.append([t, precision, recall, f1])

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print(f"Best threshold: {best_threshold:.3f}")
print(f"Best F1 score: {best_f1:.4f}")


Best threshold: 0.571
Best F1 score: 0.2827


In [29]:
final_preds = (probs >= best_threshold).astype(int)

from sklearn.metrics import classification_report
print(classification_report(y_test, final_preds))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5376
           1       0.28      0.29      0.28        93

    accuracy                           0.97      5469
   macro avg       0.63      0.64      0.63      5469
weighted avg       0.98      0.97      0.98      5469



# CATBOOST

In [31]:
path_to_repo = Path('..').resolve()
path_to_data = path_to_repo / 'data'

df_cat = pd.read_csv(path_to_data / 'data_catboost.csv')
df_cat.head()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,AGE,EXPERIENCE,bad
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,1,1,0,0,NaN,2.0,32,12,0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,1,1,0,0,NaN,2.0,32,12,0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,1,0,0,0,Security staff,2.0,58,3,0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0,1,1,Sales staff,1.0,52,8,0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0,1,1,Sales staff,1.0,52,8,0


In [32]:
y = df_cat["bad"]
X = df_cat.drop(columns=["bad", "ID"])

In [33]:
cat_features = X.select_dtypes(include=["object"]).columns.tolist()
print(cat_features)

['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE']


In [34]:
for col in cat_features:
    X[col] = X[col].fillna("missing").astype(str)

In [35]:
cat_idx = [X.columns.get_loc(c) for c in cat_features]

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [37]:
train_pool = Pool(X_train, y_train, cat_features=cat_idx)
test_pool  = Pool(X_test,  y_test,  cat_features=cat_idx)

In [38]:
model_cb = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=1500,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5,
    random_seed=42,
    border_count=128,
    auto_class_weights='Balanced',  
    verbose=200
)

In [39]:
%time model_cb.fit(train_pool, eval_set=test_pool)

0:	test: 0.4948695	best: 0.4948695 (0)	total: 72.1ms	remaining: 1m 48s
200:	test: 0.6444311	best: 0.6447997 (168)	total: 2.77s	remaining: 17.9s
400:	test: 0.6932938	best: 0.6932938 (400)	total: 5.56s	remaining: 15.3s
600:	test: 0.7064251	best: 0.7077588 (489)	total: 9.22s	remaining: 13.8s
800:	test: 0.7077985	best: 0.7083315 (770)	total: 12.3s	remaining: 10.8s
1000:	test: 0.7070001	best: 0.7083315 (770)	total: 15.5s	remaining: 7.75s
1200:	test: 0.7102707	best: 0.7106835 (1160)	total: 18.7s	remaining: 4.65s
1400:	test: 0.7113492	best: 0.7115284 (1395)	total: 22.1s	remaining: 1.56s
1499:	test: 0.7117564	best: 0.7126545 (1483)	total: 23.7s	remaining: 0us

bestTest = 0.7126545299
bestIteration = 1483

Shrink model to first 1484 iterations.
CPU times: user 2min 17s, sys: 18.1 s, total: 2min 35s
Wall time: 24 s


In [40]:
probs = model_cb.predict_proba(test_pool)[:, 1]
preds_default = (probs >= 0.5).astype(int)

In [41]:
print(classification_report(y_test, preds_default))

              precision    recall  f1-score   support

           0       0.99      0.96      0.98      7169
           1       0.16      0.44      0.24       123

    accuracy                           0.95      7292
   macro avg       0.58      0.70      0.61      7292
weighted avg       0.98      0.95      0.96      7292



In [42]:
thresholds = np.linspace(0.1, 0.9, 200)
best_t = 0.5
best_f1 = 0

for t in thresholds:
    preds_t = (probs >= t).astype(int)
    f1 = f1_score(y_test, preds_t)
    if f1 > best_f1:
        best_f1 = f1
        best_t = t

In [43]:
final_preds = (probs >= best_t).astype(int)
print(classification_report(y_test, final_preds))

              precision    recall  f1-score   support

           0       0.99      0.97      0.98      7169
           1       0.20      0.40      0.26       123

    accuracy                           0.96      7292
   macro avg       0.59      0.69      0.62      7292
weighted avg       0.98      0.96      0.97      7292



In [44]:
if 'ID' in df.columns:
    df = df.drop('ID', axis=1)

cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    df[col] = df[col].astype('category')

X = df.drop('bad', axis=1)
y = df['bad']


In [45]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [46]:
lgbm_model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,          
    is_unbalance=True,    
    random_state=42,
    n_jobs=-1,        
    importance_type='gain',  
    verbose=-1              
)

In [47]:
lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
)

LGBMClassifier(importance_type='gain', is_unbalance=True, learning_rate=0.05,
               n_estimators=1000, n_jobs=-1, random_state=42, verbose=-1)

In [48]:
print(classification_report(y_test, lgbm_model.predict(X_test)))

              precision    recall  f1-score   support

           0       0.99      0.96      0.98      7169
           1       0.17      0.46      0.25       123

    accuracy                           0.95      7292
   macro avg       0.58      0.71      0.61      7292
weighted avg       0.98      0.95      0.96      7292



In [49]:
df['INCOME_PER_PERSON'] = df['LOG_INCOME'] / df['CNT_FAM_MEMBERS']
df['AGE_YEARS'] = df['AGE'].abs() / 365
df['EXPERIENCE_YEARS'] = df['EXPERIENCE'].abs() / 365
df['EMPLOYMENT_RATIO'] = df['EXPERIENCE_YEARS'] / (df['AGE_YEARS'] + 0.001)
df['IS_UNEMPLOYED'] = (df['EXPERIENCE_YEARS'] == 0).astype(int)

In [50]:
if 'ID' in df.columns: df = df.drop('ID', axis=1)
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols: df[col] = df[col].astype('category')

X = df.drop('bad', axis=1)
y = df['bad']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [51]:
model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    is_unbalance=False, 
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

model.fit(X_train, y_train)
probs = model.predict_proba(X_test)[:, 1]

In [ ]:
thresholds = np.linspace(0.1, 0.9, 100)
best_threshold = 0.5
best_f1 = 0.0

In [53]:
for thresh in thresholds:
    y_pred_custom = (probs >= thresh).astype(int)
    score = f1_score(y_test, y_pred_custom)
    
    if score > best_f1:
        best_f1 = score
        best_threshold = thresh

In [54]:
final_preds = (probs >= best_threshold).astype(int)
print(classification_report(y_test, final_preds))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      7169
           1       0.37      0.33      0.35       123

    accuracy                           0.98      7292
   macro avg       0.68      0.66      0.67      7292
weighted avg       0.98      0.98      0.98      7292



In [ ]:
df2 = df.copy()

if 'ID' in df2.columns:
    df2 = df2.drop('ID', axis=1)

for col in df2.select_dtypes(include=['object']).columns:
    df2[col] = df2[col].astype('category')

X = df2.drop('bad', axis=1)
y = df2['bad']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [56]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "num_leaves": trial.suggest_int("num_leaves", 20, 80),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 80),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }

    model = LGBMClassifier(
        random_state=42,
        n_jobs=-1,
        **params
    )

    model.fit(X_train, y_train)

    probs = model.predict_proba(X_test)[:, 1]

    thresholds = np.linspace(0.2, 0.9, 50)
    best_f1_trial = 0

    for t in thresholds:
        preds = (probs >= t).astype(int)
        score = f1_score(y_test, preds)
        best_f1_trial = max(best_f1_trial, score)

    return best_f1_trial

In [57]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, show_progress_bar=True)

[I 2025-12-06 18:18:32,596] A new study created in memory with name: no-name-9dd8644d-de8e-4cfa-b738-f987e5c86ea9
Best trial: 0. Best value: 0.344828:   2%|▏         | 1/50 [00:04<03:41,  4.53s/it]

[I 2025-12-06 18:18:37,124] Trial 0 finished with value: 0.3448275862068966 and parameters: {'n_estimators': 772, 'learning_rate': 0.14112637322592003, 'num_leaves': 55, 'max_depth': 8, 'min_child_samples': 44, 'subsample': 0.9305834712592875, 'colsample_bytree': 0.6437518086933833, 'reg_alpha': 0.003590846864026696, 'reg_lambda': 0.3056056200525868}. Best is trial 0 with value: 0.3448275862068966.


Best trial: 0. Best value: 0.344828:   4%|▍         | 2/50 [00:09<03:51,  4.82s/it]

[I 2025-12-06 18:18:42,148] Trial 1 finished with value: 0.3381294964028777 and parameters: {'n_estimators': 1164, 'learning_rate': 0.10756147240589246, 'num_leaves': 32, 'max_depth': 11, 'min_child_samples': 62, 'subsample': 0.7131436603526072, 'colsample_bytree': 0.9178293145815333, 'reg_alpha': 0.003483161498066647, 'reg_lambda': 0.0509699931324964}. Best is trial 0 with value: 0.3448275862068966.


Best trial: 2. Best value: 0.349091:   6%|▌         | 3/50 [00:15<04:00,  5.11s/it]

...

[I 2025-12-06 18:21:47,478] Trial 47 finished with value: 0.3404255319148936 and parameters: {'n_estimators': 946, 'learning_rate': 0.1146280224039801, 'num_leaves': 45, 'max_depth': 11, 'min_child_samples': 8, 'subsample': 0.7673422923050037, 'colsample_bytree': 0.8589239318104047, 'reg_alpha': 0.06421914412434633, 'reg_lambda': 0.008430297592309448}. Best is trial 16 with value: 0.35772357723577236.


Best trial: 16. Best value: 0.357724:  98%|█████████▊| 49/50 [03:17<00:02,  2.94s/it]

[I 2025-12-06 18:21:49,626] Trial 48 finished with value: 0.21428571428571427 and parameters: {'n_estimators': 389, 'learning_rate': 0.03689137485323836, 'num_leaves': 51, 'max_depth': 8, 'min_child_samples': 76, 'subsample': 0.6418296456090036, 'colsample_bytree': 0.8976394629619964, 'reg_alpha': 0.4477991226380242, 'reg_lambda': 0.0029990115680520383}. Best is trial 16 with value: 0.35772357723577236.


Best trial: 16. Best value: 0.357724: 100%|██████████| 50/50 [03:17<00:00,  3.96s/it]

[I 2025-12-06 18:21:50,440] Trial 49 finished with value: 0.14906832298136646 and parameters: {'n_estimators': 585, 'learning_rate': 0.08263706114330871, 'num_leaves': 32, 'max_depth': 7, 'min_child_samples': 40, 'subsample': 0.7002601465141262, 'colsample_bytree': 0.7134370068131353, 'reg_alpha': 3.481376635201322, 'reg_lambda': 0.01657079545129538}. Best is trial 16 with value: 0.35772357723577236.


In [58]:
best_params = study.best_params

model_best = LGBMClassifier(
    random_state=42,
    n_jobs=-1,
    **best_params
)

In [59]:
model_best.fit(X_train, y_train)
probs = model_best.predict_proba(X_test)[:, 1]

In [60]:
thresholds = np.linspace(0.1, 0.99, 200)
best_f1 = 0
best_t = 0.5

for t in thresholds:
    preds = (probs >= t).astype(int)
    score = f1_score(y_test, preds)
    if score > best_f1:
        best_f1 = score
        best_t = t

In [61]:
final_preds = (probs >= best_t).astype(int)
print(classification_report(y_test, final_preds))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      7169
           1       0.36      0.38      0.37       123

    accuracy                           0.98      7292
   macro avg       0.67      0.69      0.68      7292
weighted avg       0.98      0.98      0.98      7292



In [63]:
import joblib

joblib.dump(model_best, "best_lgbm.pkl")

['best_lgbm.pkl']